# Day 61: Forecasting with Prophet

## Install Note
Prophet is a forecasting library originally released by Facebook (Meta), designed to be easier to use well than manually tuning an ARIMA model, especially on business time series with strong seasonal patterns (daily, weekly, yearly) and irregular events like holidays. Internally it fits an additive model — trend + seasonality + holiday effects + noise — using curve fitting rather than the autoregressive approach ARIMA uses, which tends to make it more robust to missing data and outliers with less manual tuning. Note that the package is now published as `prophet` (the older `fbprophet` name is deprecated); it has some non-Python build dependencies (like a C++ compiler for Stan) so installation can occasionally be finicky depending on your machine — that's a local setup detail, not something to "fix" in the notebook itself.

In [ ]:
# pip install prophet

## Preparing Data for Prophet
Prophet has a rigid, simple input contract: a DataFrame with exactly two columns named `ds` (the datestamp) and `y` (the numeric value to forecast) — this is different from ARIMA/statsmodels, which work with a date-indexed Series. This cell builds two years of synthetic daily data with three ingredients layered on top of each other, mirroring a realistic business metric: a gentle linear upward `trend` (+0.3 per day), a yearly seasonal cycle (a sine wave completing one full cycle per 365 days), and random daily noise. We'll hand this `df` to Prophet next and see how well it recovers the trend and seasonality on its own, without us telling it the period like we had to with `seasonal_decompose`.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2)
# 2 years of daily dates
dates = pd.date_range('2022-01-01', periods=365 * 2, freq='D')
# Combine a linear trend (+0.3/day) + a yearly seasonal sine wave (one cycle per 365 days) + daily noise
values = 200 + np.arange(len(dates)) * 0.3 + 20 * np.sin(np.arange(len(dates)) / 365 * 2 * np.pi) + np.random.normal(0, 5, len(dates))
# Prophet requires exactly these column names: 'ds' for date, 'y' for the value to forecast
df = pd.DataFrame({'ds': dates, 'y': values})

## Fitting & Forecasting
Unlike ARIMA, Prophet doesn't need you to manually specify AR/MA orders or a seasonal period — it automatically detects yearly, weekly, and (if present) daily seasonality from the timestamps, and fits a piecewise trend that can bend at detected changepoints. `model.fit(df)` learns these components from the historical data. `make_future_dataframe(periods=90)` extends the dataframe 90 days past the last observed date (keeping the same daily frequency as the training data), and `model.predict(future)` produces a forecast for every date in that extended range, including the historical dates (useful for checking fit). The result includes `yhat` (the point forecast) plus `yhat_lower`/`yhat_upper`, Prophet's uncertainty interval — wider gaps between lower and upper mean less confidence in that particular forecasted value.

In [ ]:
from prophet import Prophet

# Prophet auto-detects trend + yearly/weekly/daily seasonality from the 'ds' timestamps -- no
# manual (p, d, q) or period to set, unlike ARIMA
model = Prophet()
model.fit(df)
# Extend 90 days beyond the last training date, at the same daily frequency
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)
# yhat = point forecast; yhat_lower/yhat_upper = uncertainty interval bounds
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

## Visualizing the Forecast
Prophet ships convenience plotting methods so you don't have to build these charts manually. `model.plot(forecast)` overlays the actual observed points against the forecasted line and its uncertainty band, letting you visually check both how well the model fits history and how the forecast extends into the future — a widening uncertainty band further out is expected and healthy. `model.plot_components(forecast)` breaks the forecast down into its separate pieces (trend, yearly seasonality, weekly seasonality) plotted individually, which is genuinely useful for understanding *why* the model forecasts what it does — e.g. confirming the yearly seasonal panel shows a smooth cycle that matches the sine wave we built into the synthetic data.

In [ ]:
# fig1: observed points + forecast line + uncertainty band
fig1 = model.plot(forecast)
# fig2: forecast broken down into trend / yearly seasonality / weekly seasonality panels
fig2 = model.plot_components(forecast)